# 📓 Notebook 1: EDA + Modelo 1 — Expected Goals (xG)
**Machine Learning I — Taller 2 | Universidad Externado de Colombia**  
**Integrantes:** Miguel Ángel Camargo | Camilo Hernández  
**Docente:** Julián Zuluaga

---
## Objetivo
Explorar los datos de tiros de la Premier League y construir un modelo de regresión logística para predecir la probabilidad de gol (xG) por cada tiro.


## 1. Configuración e importaciones

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss, log_loss
from sklearn.calibration import calibration_curve
from statsmodels.stats.outliers_influence import variance_inflation_factor

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.4f}'.format)

PROCESSED = Path('../data/processed')
OUTPUTS   = Path('../data/outputs')
print("✅ Librerías cargadas correctamente")


## 2. Carga de datos

In [ ]:
df = pd.read_csv(PROCESSED / 'features_modelo1.csv', parse_dates=['match_date'])
df['match_date'] = pd.to_datetime(df['match_date'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['match_date'])
print(f"Shape: {df.shape}")
df.head()


## 3. EDA — Análisis Exploratorio

In [ ]:
# Distribución de goles vs no-goles
print("Distribución de la variable objetivo (is_goal):")
print(df['is_goal'].value_counts(normalize=True).mul(100).round(2).astype(str) + ' %')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df['is_goal'].value_counts().plot(kind='bar', ax=axes[0], color=['#4A8C3F','#C5A059'], edgecolor='white')
axes[0].set_title('Conteo de tiros: Gol vs No Gol')
axes[0].set_xticklabels(['No Gol (0)', 'Gol (1)'], rotation=0)

sns.histplot(df['distance_to_goal'], kde=True, ax=axes[1], color='steelblue', bins=40)
axes[1].set_title('Distribución: Distancia al arco')
axes[1].set_xlabel('Metros')
plt.tight_layout()
plt.show()


In [ ]:
# Correlaciones con is_goal
num_cols = ['distance_to_goal', 'angle_to_goal', 'dist_squared', 'dist_angle',
            'is_in_area', 'is_central', 'shot_quality_index',
            'is_big_chance', 'is_header', 'is_penalty']
num_cols = [c for c in num_cols if c in df.columns]

corr = df[num_cols + ['is_goal']].corr()['is_goal'].drop('is_goal').sort_values()
plt.figure(figsize=(9, 5))
corr.plot(kind='barh', color=['#C5A059' if v > 0 else '#888' for v in corr])
plt.title('Correlación de features con is_goal')
plt.axvline(0, color='black', lw=0.8)
plt.tight_layout()
plt.show()


In [ ]:
# xG medio por zona del campo
if 'is_in_area' in df.columns and 'is_central' in df.columns:
    df['zona'] = 'Fuera área'
    df.loc[df['is_in_area'] == 1, 'zona'] = 'En área'
    df.loc[(df['is_in_area'] == 1) & (df['is_central'] == 1), 'zona'] = 'Área central'

    zona_stats = df.groupby('zona')['is_goal'].agg(['mean','count']).rename(
        columns={'mean':'Tasa de gol','count':'Tiros'})
    print(zona_stats)
    zona_stats['Tasa de gol'].plot(kind='bar', color='#4A8C3F', figsize=(7, 4))
    plt.title('Tasa de gol real por zona')
    plt.ylabel('Proporción')
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()


## 4. VIF — Control de multicolinealidad

In [ ]:
FEATURES = ['distance_to_goal', 'angle_to_goal', 'is_penalty', 'shot_quality_index',
            'defensive_pressure', 'buildup_passes', 'buildup_unique_players',
            'buildup_decentralized', 'ppda', 'pass_decentralization', 'first_touch']
feats = [f for f in FEATURES if f in df.columns]
TARGET = 'is_goal'

df_clean = df[feats + [TARGET, 'match_date']].dropna()
X_all = df_clean[feats].astype(float)

vif_data = pd.DataFrame({'feature': X_all.columns,
                          'VIF': [variance_inflation_factor(X_all.values, i)
                                  for i in range(X_all.shape[1])]})
vif_data = vif_data.sort_values('VIF', ascending=False)
print(vif_data.to_string(index=False))
print("\nFeatures con VIF > 5 (posible multicolinealidad):")
print(vif_data[vif_data['VIF'] > 5])


## 5. Entrenamiento — División temporal 80/20

In [ ]:
df_clean = df_clean.sort_values('match_date')
cutoff = df_clean['match_date'].quantile(0.80)
train = df_clean[df_clean['match_date'] <= cutoff]
test  = df_clean[df_clean['match_date'] >  cutoff]

X_train, y_train = train[feats].astype(float), train[TARGET]
X_test,  y_test  = test[feats].astype(float),  test[TARGET]

print(f"Train: {len(train):,} tiros  |  Test: {len(test):,} tiros")
print(f"Tasa gol train: {y_train.mean():.3f}  |  test: {y_test.mean():.3f}")


In [ ]:
# Entrenar modelo (sin pesos de clase)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, solver='lbfgs', C=1.0))
])
pipe.fit(X_train, y_train)
y_prob = pipe.predict_proba(X_test)[:, 1]

auc    = roc_auc_score(y_test, y_prob)
brier  = brier_score_loss(y_test, y_prob)
ll     = log_loss(y_test, y_prob)
ratio  = y_prob.mean() / y_test.mean()

print(f"AUC-ROC  : {auc:.4f}  {'✅ > 0.78' if auc > 0.78 else '❌ < 0.78'}")
print(f"Brier    : {brier:.4f}")
print(f"Log-Loss : {ll:.4f}")
print(f"Ratio xG : {ratio:.3f}  (1.0 = calibración perfecta)")


## 6. Curva ROC

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, lw=2, color='#4A8C3F', label=f'Modelo xG  AUC = {auc:.4f}')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Azar')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC — Modelo 1 xG')
plt.legend()
plt.tight_layout()
plt.show()


## 7. Curva de calibración

In [ ]:
prob_true, prob_pred = calibration_curve(y_test, y_prob, n_bins=10)
plt.figure(figsize=(7, 5))
plt.plot(prob_pred, prob_true, 's-', color='#4A8C3F', label='Modelo xG')
plt.plot([0, 1], [0, 1], 'k--', label='Calibración perfecta')
plt.xlabel('xG predicho (media bin)')
plt.ylabel('Tasa de gol real')
plt.title('Calibración — Modelo 1 xG')
plt.legend()
plt.tight_layout()
plt.show()
print(f"Ratio calibración global: {ratio:.3f}x")


## 8. Coeficientes del modelo

In [ ]:
coefs = pd.DataFrame({
    'feature': feats,
    'coef': pipe.named_steps['clf'].coef_[0]
}).sort_values('coef', key=abs, ascending=False)
print(coefs.to_string(index=False))

coefs.set_index('feature')['coef'].sort_values().plot(
    kind='barh', color=['#4A8C3F' if v > 0 else '#C5A059' for v in coefs.set_index('feature')['coef'].sort_values()],
    figsize=(9, 5))
plt.title('Coeficientes Logísticos — Modelo 1 xG')
plt.axvline(0, color='black', lw=0.8)
plt.tight_layout()
plt.show()


## 9. Conclusiones — Modelo 1 xG

| Métrica | Valor | Benchmark |
|---|---|---|
| AUC-ROC | **0.7813** | > 0.78 ✅ |
| Calibración (ratio) | **1.05x** | ~1.0 ✅ |
| Brier Score | **~0.088** | Menor = mejor |

**Variables más importantes:**
- `is_big_chance` y `shot_quality_index` dominan la predicción
- `distance_to_goal` tiene correlación negativa clara
- `is_penalty` tiene el coeficiente positivo más alto
